## Environment Setup

In [ ]:
import os
import time
import math
import urllib.request
import gzip
import zipfile
import shutil
from collections import defaultdict

import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

from pyspark import SparkConf, SparkContext, Broadcast
from pyspark.sql import SparkSession
from pybloom_live import BloomFilter

import logging
import sys
import gc

print('All imports successful.')

In [2]:
def setup_logger(experiment_name):
    # Get the root logger
    logger = logging.getLogger()
    logger.setLevel(logging.INFO)
    
    # Clean up existing handlers to avoid duplicate logs or writing to old files
    if logger.hasHandlers():
        logger.handlers.clear()

    # Create file handler for the specific experiment
    file_log = logging.FileHandler(f"{experiment_name}.log")
    file_log.setFormatter(logging.Formatter('%(asctime)s - %(levelname)s - %(message)s'))
    
    # Create console handler so you can still see progress in the notebook
    console_log = logging.StreamHandler(sys.stdout)
    console_log.setFormatter(logging.Formatter('%(message)s'))

    logger.addHandler(file_log)
    logger.addHandler(console_log)
    
    return logger

# Initialize a global logger object
logger = logging.getLogger()

In [ ]:
##Key Variables
#GCS bucket path
DATA_DIR = 'gs://dataproc-staging-us-central1-327445700435-xsgaotpp/notebook-data'
CHECKPOINT_DIR = "gs://dataproc-staging-us-central1-327445700435-xsgaotpp/spark-checkpoints"
single_machine = False
MAX_VIS_NODES = 500
EXPERIMENT_1 = True
EXPERIMENT_2 = True
EXPERIMENT_3 = True
EXPERIMENT_4 = True

In [ ]:
# Single node-settings
if single_machine:
    partitions = 24
    spark = SparkSession.builder \
        .appName('CASS_Experiment_Single_Node') \
        .master("local[*]") \
        .config('spark.sql.shuffle.partitions', partitions.str) \
        .config("spark.default.parallelism", partitions.str) \
        .config('spark.executor.memory', '6g') \
        .config('spark.driver.memory', '6g') \
        .config('spark.memory.fraction', '0.6') \
        .config('spark.memory.storageFraction', '0.4') \
        .config('spark.executor.memoryOverhead', '1g') \
        .config('spark.driver.maxResultSize', '2g') \
        .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer") \
        .config("spark.python.worker.memory", "1g") \
        .config("spark.network.timeout", "800s") \
        .getOrCreate()

# #Multi-Node Settings
else:
    partitions = 48
    spark = SparkSession.builder \
        .appName('CASS_Experiment_2_Worker_Cluster') \
        .config('spark.sql.shuffle.partitions', partitions.str) \
        .config("spark.default.parallelism", partitions.str) \
        .config('spark.executor.memory', '10g') \
        .config('spark.driver.memory', '10g') \
        .config('spark.memory.fraction', '0.7') \
        .config('spark.memory.storageFraction', '0.3') \
        .config('spark.executor.memoryOverhead', '2g') \
        .config('spark.driver.maxResultSize', '4g') \
        .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer") \
        .config("spark.python.worker.memory", "2g") \
        .config("spark.network.timeout", "800s") \
        .getOrCreate()
sc = spark.sparkContext
print(f'Spark version: {spark.version}')
print(f'Running with {sc.defaultParallelism} cores')

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/03 04:32:59 INFO SparkEnv: Registering MapOutputTracker
26/04/03 04:32:59 INFO SparkEnv: Registering BlockManagerMaster
26/04/03 04:32:59 INFO SparkEnv: Registering BlockManagerMasterHeartbeat
26/04/03 04:32:59 INFO SparkEnv: Registering OutputCommitCoordinator


Spark version: 3.5.3
Running with 48 cores


## Download and Read Data Functions

In [ ]:
##Google services optomized download_dataset
def download_dataset(url, foldername):
    """
    Download the data from url and unpack.
    """
    gcs_path = f"{DATA_DIR}/{foldername}"
    # Use gsutil to check if it exists in GCS already
    exists = os.system(f"gsutil -q stat {gcs_path}") == 0
    
    if not exists:
        logger.info(f"Downloading {url} to GCS...")
        # Download locally to /tmp first
        local_tmp = f"/tmp/{foldername}.gz"
        os.system(f"curl -L {url} -o {local_tmp}")
        
        # If it's a .gz, unzip and stream to GCS
        if url.endswith('.gz'):
            os.system(f"gunzip -c {local_tmp} | gsutil cp - {gcs_path}")
        else:
            os.system(f"gsutil cp {local_tmp} {gcs_path}")
    
    return gcs_path

##Load the edge list
def load_edge_list(filepath, filename, delimiter=','):
    """
    Load an undirected edge list from a CSV or TXT file.
    Skips comment lines starting with '#'.
    Returns an RDD of (src, dst) tuples in canonical form (src < dst).
    """
    logger.info('loading edge list')
    if filename.endswith('.txt'):
      raw = sc.textFile(filepath)
      logger.info('file loaded')
      edges = raw \
          .filter(lambda line: not line.startswith('#') and line.strip() != '') \
          .map(lambda line: line.strip().split()) \
          .filter(lambda parts: len(parts) >= 2) \
          .map(lambda parts: (int(parts[0]), int(parts[1]))) \
          .filter(lambda e: e[0] != e[1])  # remove self-loops
      logger.info('edges formed')
      # Canonical form: ensure src < dst to avoid duplicates
      canonical = edges.map(lambda e: (min(e), max(e))).distinct()
      return canonical
    
    elif filename.endswith('.csv'):
      raw = sc.textFile(f"{filepath}/{filename}")
      logger.info('file loaded')
      edges = raw \
          .filter(lambda line: not line.startswith('#') and line.strip() != '') \
          .map(lambda line: line.strip().split(delimiter)) \
          .filter(lambda parts: len(parts) >= 2) \
          .map(lambda parts: (int(parts[0]), int(parts[1]))) \
          .filter(lambda e: e[0] != e[1])  # remove self-loops
      logger.info('edges formed')
      # Canonical form: ensure src < dst and remove duplicates
      canonical = edges.map(lambda e: (min(e), max(e))).distinct()
      return canonical

## CASS Algorithm Implementation

In [ ]:
def compute_degrees(canonical_edges):
    """
    Compute degree of each node.
    The degree includes the node itself: |Nei(v)| = degree(v) + 1
    as per the paper's definition: Nei(v) = {w in V | (v,w) in E} union {v}
    """
    # Since edges are canonical (src < dst), count both directions
    degree_rdd = canonical_edges.flatMap(lambda e: [(e[0], 1), (e[1], 1)]) \
        .reduceByKey(lambda a, b: a + b)
    
    logger.info("Collecting degree map to driver (this may take a moment)...")
    degree_map = degree_rdd.collectAsMap()
    return degree_map # This will be passed to sc.broadcast()  # actual graph degree: Nei size = degree + 1

def build_bloom_filter(canonical_edges, num_edges, false_positive_rate=0.3):
    """
    Build a Bloom filter containing all edges.
    Each edge is stored as a string 'src:dst' in canonical form (src < dst).
    """
    bf = BloomFilter(capacity=num_edges, error_rate=false_positive_rate)
    # Gather edges to driver to build the filter
    logger.info("Building Bloom Filter...")
    #.toLocalIterator() used instead of .collect(). Fetches data one partition at a time.
    #Slower for small datasets, but safer for the Out of Memory error (OOM) because we only 
    #need enough memory for the largest partition. 
    for src, dst in canonical_edges.toLocalIterator():
        key = f'{min(src,dst)}:{max(src,dst)}'
        bf.add(key)
    return bf

def shuffle_selection(edges_rdd):
    """
    Determine optimal join direction.
    Returns a directed edge RDD keyed by the side with MORE distinct keys
    (which produces fewer intermediate results per key on average).
    """

    # Count distinct source and destination vertices
    n_src = edges_rdd.map(lambda e: e[0]).distinct().count()
    n_dst = edges_rdd.map(lambda e: e[1]).distinct().count()

    logger.info(f'  Distinct sources: {n_src:,}, Distinct destinations: {n_dst:,}')

    # Use the direction with more distinct keys (= fewer pairs per key)
    if n_src >= n_dst:
        logger.info('   Using original direction (keyed by source)')
        return edges_rdd  # keyed by src
    else:
        logger.info('   Using reversed direction (keyed by destination)')
        return edges_rdd.map(lambda e: (e[1], e[0]))  # keyed by dst

def find_triangles_and_compute_similarity(directed_edges, canonical_edges, bloom_filter, degree_bc):
    """
    Find all triangles and compute structure similarity for each edge.

    Structure similarity(v, w) = |Nei(v) intersect Nei(w)| / sqrt(|Nei(v)| * |Nei(w)|)

    where |Nei(v) intersect Nei(w)| = (# triangles containing edge (v,w)) + 2
    (the +2 accounts for v and w themselves being in each other's neighborhood,
    plus the self-inclusion: Nei(v) includes v)
    """
    #Create the bloom filter and broadcast it to nodes in the network. 
    bf = bloom_filter  
    broadcast_bf = sc.broadcast(bf)
    #Try to repartition to rebalance the big dataset for optimal performance
    directed_edges = directed_edges.repartition(partitions)

    # Step 1: First inner join - find candidate triangles
    # Self-join on key: (v, w1) JOIN (v, w2) becomes candidate (v, w1, w2) sharing vertex v
    # We only keep w1 < w2 to avoid duplicates
    candidates = directed_edges.join(directed_edges) \
        .filter(lambda x: x[1][0] < x[1][1]) \
        .map(lambda x: (x[0], x[1][0], x[1][1]))

    # Step 2: Bloom filter - prune candidates
    #Use broadcast variables. These are read only shared varibles cached on each worker node
    #rather than sent accross the network with each task.
    def bloom_check(triple):
        v, w1, w2 = triple
        key = f'{min(w1,w2)}:{max(w1,w2)}'
        return key in broadcast_bf.value

    filtered_candidate_edges = candidates.filter(bloom_check)

    # STEP 3: Second Join (Existence Check)
    # We must join with the canonical edge set to confirm the triangle.
    # We map candidates to ((w1, w2), v) and join with ((u, v), 1)
    formatted_edges = canonical_edges.map(lambda e: ((min(e[0],e[1]), max(e[0],e[1])), 1))

    confirmed_triangles = filtered_candidate_edges \
        .map(lambda t: ((min(t[1], t[2]), max(t[1], t[2])), t[0])) \
        .join(formatted_edges) \
        .map(lambda x: (x[0][0], x[0][1], x[1][0])) # (w1, w2, common_v)

    # Step 4: Count shared neighbors per edge
    # A triangle (v, w1, w2) means:
    # edges (v,w1), (v,w2), (w1,w2) each gain +1 shared neighbor.
    triangle_edge_counts = confirmed_triangles.flatMap(lambda x: [
            ((min(x[0], x[1]), max(x[0], x[1])), 1),  # edge (w1, w2)
            ((min(x[0], x[2]), max(x[0], x[2])), 1),  # edge (w1, v)
            ((min(x[1], x[2]), max(x[1], x[2])), 1),  # edge (w2, v)
        ]).reduceByKey(lambda a, b: a + b)

    # Step 5: Compute structure similarity
    deg = degree_bc  # broadcast variable

    def calc_similarity(edge_and_count):
        (v, w), triangle_count = edge_and_count
        # |Nei(v) intersect Nei(w)| = triangle_count + 2
        # (+2 because v is element of Nei(v) intersect Nei(w) and w element of Nei(v) intersect Nei(w))
        shared = triangle_count + 2
        # |Nei(v)| = degree(v) + 1 (includes self)
        nei_v = deg.value.get(v, 0) + 1
        nei_w = deg.value.get(w, 0) + 1
        sim = shared / math.sqrt(nei_v * nei_w)
        return ((v, w), min(sim, 1.0))  # cap at 1.0

    similarity_from_triangles = triangle_edge_counts.map(calc_similarity)

    # Edges with 0 triangles still need similarity = 2 / sqrt(|Nei(v)|*|Nei(w)|)
    # (because shared = 0 + 2 = 2, i.e. only v and w themselves)
    all_canonical = canonical_edges \
        .map(lambda e: (min(e[0],e[1]), max(e[0],e[1]))) \
        .distinct()

    edges_with_triangles = similarity_from_triangles.map(lambda x: (x[0], x[1]))

    # Left outer join to find edges with no triangles
    all_with_flag = all_canonical.map(lambda e: (e, None)).leftOuterJoin(
        edges_with_triangles.map(lambda x: (x[0], x[1]))
    )

    def fill_missing_similarities(item):
        edge, (_, sim) = item
        if sim is not None:
            return (edge, sim)
        v, w = edge
        nei_v = deg.value.get(v, 0) + 1
        nei_w = deg.value.get(w, 0) + 1
        return (edge, 2.0 / math.sqrt(nei_v * nei_w))

    all_similarities = all_with_flag.map(fill_missing_similarities)

    return all_similarities, broadcast_bf

def reconstruct_network(similarity_rdd, epsilon=0.7, m=2):
    """
    Reconstruct the network:
    1. Prune edges with structure similarity < epsilon
    2. Label core/non-core based on neighbor count >= m
    3. Remove edges between two non-core nodes

    Returns: pruned edge RDD and core set
    """
    logger.info(f'  Parameters: epsilon={epsilon}, m={m}')

    # Step 1: Prune edges below epsilon
    pruned_edges = similarity_rdd \
        .filter(lambda x: x[1] >= epsilon) \
        .map(lambda x: x[0])  # just keep (src, dst)
    pruned_edges.cache()

    # Step 2: Compute degrees in pruned graph and label core/non-core
    pruned_degrees = pruned_edges \
        .flatMap(lambda e: [(e[0], 1), (e[1], 1)]) \
        .reduceByKey(lambda a, b: a + b)

    core_nodes = pruned_degrees \
        .filter(lambda x: x[1] >= m) \
        .map(lambda x: x[0])

    # Convert to a dictionary for faster lookup before broadcasting
    core_map = core_nodes.map(lambda x: (x, True)).collectAsMap()
    core_bc = sc.broadcast(core_map)

    # Step 3: Remove edges between two non-core nodes
    # Keep edge if at least one endpoint is core
    def has_core_endpoint(edge):
        cs = core_bc.value
        return edge[0] in cs or edge[1] in cs

    final_edges = pruned_edges.filter(has_core_endpoint)
    final_edges.cache()

    return final_edges, core_map, core_bc

def find_components_spark(edges_rdd, max_iter=100): 
    sc = edges_rdd.context
    sc.setCheckpointDir(CHECKPOINT_DIR) 
    num_partitions = edges_rdd.getNumPartitions()

    adj = edges_rdd.flatMap(lambda e: [(e[0], e[1]), (e[1], e[0])]) \
                   .partitionBy(num_partitions) \
                   .cache()

    nodes = adj.keys().distinct()
    labels = nodes.map(lambda v: (v, v)).partitionBy(num_partitions).cache()

    for i in range(max_iter):
        propagated = adj.join(labels).map(lambda x: (x[1][0], x[1][1]))
        new_labels = labels.union(propagated) \
                           .reduceByKey(min, numPartitions=num_partitions)
        new_labels.localCheckpoint()
        changed = not labels.join(new_labels).filter(lambda x: x[1][0] != x[1][1]).isEmpty()
        labels.unpersist()
        labels = new_labels.cache()
        logger.info(f'    Iteration {i+1}')
        if not changed:
            logger.info(f'    Converged!')
            break
    return labels

## Analysis Functions

In [ ]:
def analyze_clusters(cluster_labels, epsilon, num_nodes, m):
  cluster_sizes = cluster_labels \
      .map(lambda x: (x[1], 1)) \
      .reduceByKey(lambda a, b: a + b) \
      .sortBy(lambda x: -x[1]) \
      .collect()

  n_clusters = len(cluster_sizes)
  n_clustered_nodes = sum(s for _, s in cluster_sizes)
  single_node_clusters = sum(1 for _, s in cluster_sizes if s == 1)
  multi_node_clusters = n_clusters - single_node_clusters

  logger.info(f'\n=== Clustering Results (epsilon={epsilon}, m={m}) ===')
  logger.info(f'Total clusters: {n_clusters:,}')
  logger.info(f'Multi-node clusters (size >= 2): {multi_node_clusters:,}')
  logger.info(f'Single-node clusters: {single_node_clusters:,}')
  logger.info(f'Nodes in clusters: {n_clustered_nodes:,} / {num_nodes:,}')
  logger.info(f'Remaining node ratio: {1 - n_clustered_nodes/num_nodes:.4f}')

  logger.info(f'\nTop 20 largest clusters:')
  for cid, size in cluster_sizes[:20]:
      logger.info(f'  Cluster {cid}: {size:,} nodes')
        
    
def compute_avg_normalized_cut(edges_rdd, cluster_labels):
    """
    Compute average normalized cut between all pairs of clusters.
    Uses a sampling approach for large numbers of clusters.
    """
    # Collect cluster assignments
    node_to_cluster = dict(cluster_labels.collect())
    node_cluster_bc = sc.broadcast(node_to_cluster)

    # For each edge, determine if it's intra-cluster or inter-cluster
    def classify_edge(edge):
        nc = node_cluster_bc.value
        v, w = edge
        cv = nc.get(v, v)
        cw = nc.get(w, w)
        return (cv, cw)

    # Count edges per cluster and between clusters
    edge_classes = edges_rdd.map(lambda e: (classify_edge(e), 1)) \
        .reduceByKey(lambda a, b: a + b) \
        .collect()

    # Build counts
    intra = defaultdict(int)  # edges within cluster
    inter = defaultdict(int)  # edges between cluster pairs
    total_edges_per_cluster = defaultdict(int)  # total edges touching cluster

    for (ca, cb), count in edge_classes:
        if ca == cb:
            intra[ca] += count
            total_edges_per_cluster[ca] += count
        else:
            pair = (min(ca, cb), max(ca, cb))
            inter[pair] += count
            total_edges_per_cluster[ca] += count
            total_edges_per_cluster[cb] += count

    # Compute normalized cut for each pair of adjacent clusters
    ncut_values = []
    for (ca, cb), cut_edges in inter.items():
        ea = total_edges_per_cluster.get(ca, 0)
        eb = total_edges_per_cluster.get(cb, 0)
        if ea > 0 and eb > 0:
            ncut = cut_edges / ea + cut_edges / eb
            ncut_values.append(ncut)

    if ncut_values:
        avg_ncut = sum(ncut_values) / len(ncut_values)
    else:
        avg_ncut = 0.0

    return avg_ncut, len(ncut_values)

def report_normalized_cut(edges_rdd, cluster_labels): 
  logger.info('Computing average normalized cut...')
  t0 = time.time()
  avg_ncut, n_pairs = compute_avg_normalized_cut(edges_rdd, cluster_labels)
  ncut_time = time.time() - t0
  logger.info(f'Average Normalized Cut: {avg_ncut:.4f} (over {n_pairs:,} cluster pairs)')
  logger.info(f'Computed in {ncut_time:.2f}s')

## Cleanup Function

In [ ]:
def cleanup_memory(rdds_to_unpersist=[]):
    """
    Clear Spark caches, Python garbage, and JVM heap.
    """
    logger.info("--- Starting Memory Cleanup ---")
    
    # 1. Unpersist specific RDDs passed in
    for rdd in rdds_to_unpersist:
        try:
            rdd.unpersist()
        except:
            pass
            
    try:
        sc._jsc.sc().getPersistentRDDs().values().foreach(
            sc._jvm.pyspark.InternalAccumulator.getUnpersistFunc()
        )
        logger.info("  Successfully signaled Spark to unpersist all RDDs.")
    except Exception as e:
        logger.warning(f"  Note: Could not bulk clear RDDs: {e}")
    
    # 3. Python Garbage Collection
    gc.collect()
    
    # 4. JVM Garbage Collection
    sc._jvm.System.gc()
    
    logger.info("--- Cleanup Complete ---")

## Run Experiment Helper Functions

In [ ]:
def run_cass(filepath, filename, eps_values, m_values, epsilon=0.7, m=2, fp_rate=0.3,):
    """
    Complete CASS pipeline on an edge list file.

    Parameters:
        filepath: path to edge list (tab or space separated, # comments)
        epsilon: structure similarity threshold (default 0.7)
        m: minimum neighbors for core node (default 2)
        fp_rate: Bloom filter false positive rate (default 0.3)

    Returns:
        cluster_labels: RDD of (node_id, cluster_id)
        similarity_rdd: RDD of ((src, dst), similarity) — for reuse
    """
    logger.info(f'\n{"="*60}')
    logger.info(f'CASS: ε={epsilon}, m={m}, fp_rate={fp_rate}')
    logger.info(f'File: {filepath}')
    logger.info(f'{"="*60}')

    t_start = time.time()
    logger.info('timer started')

    # Load
    edges = load_edge_list(filepath, filename)
    logger.info('edges loaded')
    edges.cache()
    ne = edges.count()
    logger.info('edges counted')
    nn = edges.flatMap(lambda e: [e[0], e[1]]).distinct().count()
    logger.info(f'Loaded: {nn:,} nodes, {ne:,} edges')

    # Degrees
    deg_map = compute_degrees(edges)
    deg_bc = sc.broadcast(deg_map)

    # Bloom filter
    bf = build_bloom_filter(edges, ne, fp_rate)

    # Shuffle selection + directed edges
    directed = shuffle_selection(edges)
    directed.cache()
    directed.foreach(lambda x: None)

    # Structure similarity
    sim, bf_bc = find_triangles_and_compute_similarity(directed, edges, bf, deg_bc)
    sim.cache()
    sim.foreach(lambda x: None)

    # Reconstruct
    final_e, cores, core_bc = reconstruct_network(sim, epsilon, m)
    sim.unpersist() #garbage collection
    
    # Connected components
    labels = find_components_spark(final_e)
    labels.cache()
    # labels.count()
    labels.foreach(lambda x: None)

    t_total = time.time() - t_start

    #------- Report -------#
    logger.info('Analyze Clusters')
    analyze_clusters(
        cluster_labels=labels,
        epsilon=epsilon,
        num_nodes=nn,
        m=m
    )

    sizes = labels.map(lambda x: (x[1], 1)).reduceByKey(lambda a, b: a + b).collect()
    multi = len([s for _, s in sizes if s >= 2])
    avg_ncut_val, _ = compute_avg_normalized_cut(edges, labels)

    logger.info(f'\nResults: {multi} clusters (size>=2), Avg N-cut={avg_ncut_val:.4f}, Time={t_total:.1f}s')


    return labels, deg_bc, core_bc, bf_bc

def run_experiment(exp_name, url, folder, filename, eps_values, m_values, epsilon=0.7, m=2, fp_rate=0.3):
    setup_logger(exp_name)  
    logger.info(f"Starting Experiment: {exp_name}")
    ##Download dataset
    data_path = download_dataset(
      url=url,
      foldername=folder,
  )
    logger.info('\nDataset ready.')

    labels, d_bc, c_bc, bf_bc = run_cass(
      filepath=data_path,
      filename=filename,
      epsilon=epsilon,
      m=m,
      fp_rate=fp_rate,
      eps_values=eps_values,
      m_values=m_values
  )
    d_bc.unpersist() # Frees broadcast memory on workers
    c_bc.unpersist()
    bf_bc.unpersist()
    cleanup_memory([labels])

## Experiment Parameters

In [ ]:
## Datasets to analyze
experiments = {
    'small': ['https://snap.stanford.edu/data/email-Eu-core.txt.gz', 'email-Eu-core.txt', 'email-Eu-core.txt'],
    'medium': ['https://snap.stanford.edu/data/email-Enron.txt.gz', 'email-Enron.txt', 'email-Enron.txt'],
    'small_large': ['https://snap.stanford.edu/data/web-Stanford.txt.gz', 'web-Stanford.txt', 'web-Stanford.txt'],
    'large_large': ['https://snap.stanford.edu/data/web-Google.txt.gz', 'web-Google.txt', 'web-Google.txt'],
    }
print(f"Running:\n    Experiment 1: {EXPERIMENT_1}\n    Experiment 2: {EXPERIMENT_2}\n    Experiment 3: {EXPERIMENT_3}\n    Experiment 4: {EXPERIMENT_4}\n")

In [ ]:
#Experiment 1
url = experiments['small'][0]
folder = experiments['small'][1]
filename = experiments['small'][2]

if EXPERIMENT_1:

    run_experiment(
        exp_name='2_node_small',
        url=url,
        folder=folder,
        filename=filename,
        eps_values=eps_values,
        m_values=m_values
    )
os.system(f"gsutil cp *.log {DATA_DIR}/logs/")

In [ ]:
#Experiment 2
url = experiments['medium'][0]
folder = experiments['medium'][1]
filename = experiments['medium'][2]

if EXPERIMENT_2:
    run_experiment(
        exp_name='2_node_medium',
        url=url,
        folder=folder,
        filename=filename,
        eps_values=eps_values,
        m_values=m_values
    )
os.system(f"gsutil cp *.log {DATA_DIR}/logs/")

In [ ]:
#Experiment 3
url = experiments['small_large'][0]
folder = experiments['small_large'][1]
filename = experiments['small_large'][2]

if EXPERIMENT_3:
    run_experiment(
        exp_name='2_node_small_large',
        url=url,
        folder=folder,
        filename=filename,
        eps_values=eps_values,
        m_values=m_values
    )
os.system(f"gsutil cp *.log {DATA_DIR}/logs/")

In [ ]:
#Experiment 4

url = experiments['large_large'][0]
folder = experiments['large_large'][1]
filename = experiments['large_large'][2]

if EXPERIMENT_4:
    run_experiment(
        exp_name='2_node_large_large',
        url=url,
        folder=folder,
        filename=filename,
        eps_values=eps_values,
        m_values=m_values
    )
os.system(f"gsutil cp *.log {DATA_DIR}/logs/")